# Orbit Wars RL Observation Sistemi

Bu notebook, raw Kaggle Orbit Wars environment observation'ının RL policy için fixed-shape tensorlara nasıl dönüştürüldüğünü açıklar.

Ana kaynak dosyalar:

- `obs_builder.py`: Observation dataclass, planet/fleet/global feature extraction, edge bias ve action decode.
- `feature_extraction.py`: raw env observation parse işlemleri (`Planet`, `Fleet`).
- `navigation.py`: fleet target prediction ve launch angle hesaplamaları.
- `model.py` / `model_planet_only.py`: observation tensorlarını tüketen policy modelleri.


## 1. Observation Builder'ın Amacı

Environment observation'ı değişken sayıda planet ve fleet içerir. Neural network ise sabit shape ister. `build_observation(...)` bu değişken raw state'i maskeli, padded tensorlara çevirir.

Çıktı `Observation` dataclass'ıdır:

```text
planet_feats : (P_max, PLANET_FEATURE_DIM) float32
planet_mask  : (P_max,) bool
fleet_feats  : (F_max, FLEET_FEATURE_DIM) float32
fleet_mask   : (F_max,) bool
global_feats : (GLOBAL_FEATURE_DIM,) float32
edge_bias    : (T, T, EDGE_FEATURE_DIM) float32
source_mask  : (P_max,) bool
planet_ids   : (P_max,) int32
fleet_ids    : (F_max,) int32
meta         : dict
```

Burada:

```text
P_max = 48
F_max = 64
T = P_max + F_max + 1
```

Son token global token'dır.


## 2. Neden Set/Transformer Formatı?

Orbit Wars state'i doğal olarak bir set yapısına sahiptir:

- Planetler sıralı değil.
- Fleetler sıralı değil.
- Her oyun farklı sayıda fleet içerebilir.

Bu yüzden observation builder planet ve fleetleri fixed slotlara koyar, gerçek slotları maskelerle belirtir. Transformer, tokenlar arası ilişkileri attention ile öğrenir.

Model token layout'u:

```text
[planet_0, ..., planet_{P_max-1}, fleet_0, ..., fleet_{F_max-1}, GLOBAL]
```

`planet_only` varyantında fleet tokenlar modelin encoderına girmez; ama observation hala genel formatı üretir.


## 3. Raw Env Observation

Raw observation temel olarak şunları içerir:

```text
planets
fleets
initial_planets
comet_planet_ids / comets
angular_velocity
step
player
```

`feature_extraction.parse_planets` raw planet satırlarını `Planet` dataclass'ına çevirir. Benzer şekilde `parse_fleets` raw fleetleri `Fleet` dataclass'ına çevirir.

Observation builder oyuncu-perspektiflidir: aynı board state, player `0` için farklı; player `1`, `2`, `3` için farklı feature encoding'e sahip olur. Özellikle owner one-hot değerleri player'a göre hesaplanır.


## 4. Planet Feature Tensor

Planet feature dimension şu an:

```text
PLANET_FEATURE_DIM = 34
```

Feature indeksleri `obs_builder.py` içinde `PF_IDX_*` constantlarıyla tanımlı. Hard-coded integer kullanmak yerine bu constantlar kullanılmalı.

Planet feature grupları:

| Grup | Featurelar | Amaç |
|---|---|---|
| Ownership | self / enemy_0 / enemy_1 / enemy_2 / neutral one-hot | 4 oyunculu oyunda 3 düşmanı ayrı temsil eder |
| Economy | ships, production, radius | Capture/defense değeri |
| Geometry | sun-relative radius, cos theta, sin theta | Orbital konum |
| Dynamics | static/rotating/comet, omega | Hareket bilgisi |
| Ship scale | ships, log_ships | Garrison büyüklüğünü lineer ve log ölçekte verir |
| Incoming buckets | self/enemy incoming by ETA bucket | Yaklaşan fleet tehdidi |
| Threat summaries | total incoming, min ETA, largest incoming | Defense kararı |
| Value estimates | capture_value, defense_deficit | Reward-aware state feature |

`ships_to_cap` artık ayrı feature değildir; target üzerindeki `ships` değeri zaten aynı bilgiyi doğrudan verdiği için kaldırıldı.

## 5. Planet Feature İndeksleri

Mevcut sıra:

| Index | Constant | Değer | Normalizasyon |
|---:|---|---|---|
| 0 | `PF_IDX_OWNER_SELF` | Planet bana ait mi | 0/1 |
| 1 | `PF_IDX_OWNER_ENEMY_0` | Player-relative ilk düşman | 0/1 |
| 2 | `PF_IDX_OWNER_ENEMY_1` | Player-relative ikinci düşman | 0/1 |
| 3 | `PF_IDX_OWNER_ENEMY_2` | Player-relative üçüncü düşman | 0/1 |
| 4 | `PF_IDX_OWNER_NEUTRAL` | Neutral mı | 0/1 |
| 5 | `PF_IDX_SHIPS` | Garrison ship | `/100` |
| 6 | `PF_IDX_PRODUCTION` | Production | `/5` |
| 7 | `PF_IDX_RADIUS` | Radius | `/5` |
| 8 | `PF_IDX_R_SUN` | Sun'a uzaklık | `/50` |
| 9 | `PF_IDX_COS_THETA` | cos(theta) | raw |
| 10 | `PF_IDX_SIN_THETA` | sin(theta) | raw |
| 11 | `PF_IDX_IS_STATIC` | Static planet | 0/1 |
| 12 | `PF_IDX_IS_ROTATING` | Rotating planet | 0/1 |
| 13 | `PF_IDX_IS_COMET` | Comet | 0/1 |
| 14 | `PF_IDX_OMEGA` | Angular velocity | raw if rotating else 0 |
| 15 | `PF_IDX_LOG_SHIPS` | log(ships+1) | `/log(1000)` |
| 16-19 | incoming self buckets | Friendly incoming ships by ETA | `/100` |
| 20-23 | incoming enemy buckets | Enemy incoming ships by ETA | `/100` |
| 24 | `PF_IDX_INCOMING_SELF_TOTAL` | Toplam friendly incoming | `/100` |
| 25 | `PF_IDX_INCOMING_ENEMY_TOTAL` | Toplam enemy incoming | `/100` |
| 26 | `PF_IDX_INCOMING_SELF_MIN_ETA` | En yakın friendly arrival | capped `/50` |
| 27 | `PF_IDX_INCOMING_ENEMY_MIN_ETA` | En yakın enemy arrival | capped `/50` |
| 28 | `PF_IDX_INCOMING_ENEMY_SOONEST_SHIPS` | En erken enemy fleet ships | `/100` |
| 29 | `PF_IDX_INCOMING_ENEMY_LARGEST_SHIPS` | En büyük enemy incoming | `/100` |
| 30 | `PF_IDX_INCOMING_ENEMY_LARGEST_ETA` | En büyük incoming enemy fleet ETA | capped `/50` |
| 31 | `PF_IDX_INCOMING_NET_THREAT` | enemy_total - ships - self_total | `/100` |
| 32 | `PF_IDX_CAPTURE_VALUE` | production * remaining_fraction | `/50` |
| 33 | `PF_IDX_DEFENSE_DEFICIT` | enemy threat deficit | `/100` |

In [ ]:
PLANET_FEATURES = [
    'owner_self', 'owner_enemy_0', 'owner_enemy_1', 'owner_enemy_2', 'owner_neutral',
    'ships/100', 'production/5', 'radius/5', 'r_sun/50', 'cos_theta', 'sin_theta',
    'is_static', 'is_rotating', 'is_comet', 'omega', 'log_ships',
    'incoming_self_0_5', 'incoming_self_5_15', 'incoming_self_15_30', 'incoming_self_30_plus',
    'incoming_enemy_0_5', 'incoming_enemy_5_15', 'incoming_enemy_15_30', 'incoming_enemy_30_plus',
    'incoming_self_total', 'incoming_enemy_total', 'incoming_self_min_eta', 'incoming_enemy_min_eta',
    'enemy_soonest_ships', 'enemy_largest_ships', 'enemy_largest_eta',
    'incoming_net_threat', 'capture_value', 'defense_deficit',
]
for i, name in enumerate(PLANET_FEATURES):
    print(f'{i:02d}: {name}')

## 6. Fleet Feature Tensor

Fleet feature dimension:

```text
FLEET_FEATURE_DIM = 11
```

Fleet featureları:

| Index | Değer | Normalizasyon |
|---:|---|---|
| 0 | fleet bana ait mi | 0/1 |
| 1 | fleet enemy'ye ait mi | 0/1 |
| 2 | ships | `/100` |
| 3 | log_ships | `/log(1000)` |
| 4 | speed | `/DEFAULT_MAX_SHIP_SPEED` |
| 5 | cos(angle) | raw |
| 6 | sin(angle) | raw |
| 7 | x coordinate | `x / BOARD_SIZE - 0.5` |
| 8 | y coordinate | `y / BOARD_SIZE - 0.5` |
| 9 | predicted ETA | capped `/50` |
| 10 | has prediction | 0/1 |

Target prediction `navigation.predict_all_fleet_targets` ile yapılır. Bu, incoming threat bucketlarını ve fleet ETA featurelarını doldurur.


## 7. Fleet Truncation ve Priority

Eğer raw observation `F_MAX_DEFAULT=64`'ten fazla fleet içerirse, builder en önemli fleetleri seçer. Priority yaklaşık şu sıradır:

1. Enemy fleet benim planetlerime geliyorsa
2. Friendly fleet benim planetlerime geliyorsa
3. Diğer enemy fleetler
4. Diğer friendly fleetler
5. Nötr/diğer

Aynı priority içinde düşük ETA ve yüksek ship önceliklidir. Bu sayede defense açısından kritik fleetler modelden düşmez.


## 8. Global Features

Global feature dimension:

```text
GLOBAL_FEATURE_DIM = 14
```

Featurelar:

| Index | Değer | Normalizasyon |
|---:|---|---|
| 0 | turn | `/200` |
| 1 | angular velocity | raw |
| 2 | my planet count | `/25` |
| 3 | enemy planet count | `/25` |
| 4 | neutral planet count | `/25` |
| 5 | my total ships | `/100` |
| 6 | enemy total ships | `/100` |
| 7 | my total production | `/50` |
| 8 | enemy total production | `/50` |
| 9 | fleet count | `/25` |
| 10 | enemy incoming to my planets | `/100` |
| 11 | friendly incoming to my planets | `/100` |
| 12 | my capture value estimate | `/5000` |
| 13 | airborne ship advantage | `/100` |

Global token, transformer'da board-level context sağlamak için kullanılır. Gemi sayısı içeren global featurelar da planet ship feature ile aynı ölçeğe çekildi.

## 9. Masks

Observation üç önemli mask üretir.

### `planet_mask`

Shape: `(P_max,)`. Gerçek planet slotları `True`, padding slotları `False`.

Model target logits içinde padded planetleri target olarak maskelemek için kullanılır.

### `fleet_mask`

Shape: `(F_max,)`. Gerçek fleet slotları `True`, padding slotları `False`.

Full model attention key padding mask içinde kullanır.

### `source_mask`

Shape: `(P_max,)`. Agent'ın launch yapabileceği planetleri belirtir.

Kural:

```text
source_mask[i] = planet.owner == player and planet.ships >= min_ships_to_launch
```

Model source logits'i bu maskeyle invalid source slotlarını `-inf` yapar.


## 10. Edge Bias Tensor

`edge_bias` shape'i:

```text
(T, T, EDGE_FEATURE_DIM)
EDGE_FEATURE_DIM = 6
```

Her token çifti `(i, j)` için geometrik ilişki verir. Modelde bu tensor lineer projeksiyonla per-head additive attention bias'a çevrilir.

Edge featureları:

| Index | Değer | Anlam |
|---:|---|---|
| 0 | distance / BOARD_SIZE | İki token arası mesafe |
| 1 | dx / BOARD_SIZE | j'nin i'ye göre x farkı |
| 2 | dy / BOARD_SIZE | j'nin i'ye göre y farkı |
| 3 | sin(delta_phi) | Sun merkezli angular fark |
| 4 | cos(delta_phi) | Sun merkezli angular fark |
| 5 | crosses_sun | Segment sun'a çok yakın geçiyor mu |

Diagonal sıfırlanır. Aksi halde `cos(0)=1` self-attention'a sabit bias sızdırırdı.


## 11. Incoming Threat Aggregation

Incoming fleetler her planet için ETA bucketlarına ayrılır. Bucketlar:

```text
[0, 5], (5, 15], (15, 30], 30+
```

Hem self incoming hem enemy incoming ayrı tutulur. Ek olarak:

- toplam incoming
- minimum ETA
- en erken enemy fleet ships
- en büyük enemy incoming ships
- en büyük enemy incoming ETA
- net threat
- defense deficit

hesaplanır.

Bu featurelar özellikle şu kararları destekler:

- Source planet'ten launch yapmak güvenli mi?
- No-op yapıp savunmak mı daha iyi?
- Target'a zaten friendly fleet gidiyor mu?
- Enemy incoming yüzünden planet düşmek üzere mi?


## 12. Action Decode

Policy output doğrudan env action değildir. Model şu discrete action bileşenlerini üretir:

```text
src_slot
tgt_slot
ship_bin
is_noop
```

`decode_action_with_launch(...)` bunları environment action formatına çevirir:

```text
[src_planet_id, angle, ships]
```

Ship bin fractionları:

```text
DEFAULT_SHIP_BINS = (0.1, 0.25, 0.5, 0.75, 1.0)
```

Ships hesabı:

```text
ships = max(1, int(src.available_ships * ship_fraction))
```

Angle `navigation.compute_launch` ile hesaplanır. Eğer launch geometrik olarak mümkün değilse `None` döner ve action invalid sayılabilir.


In [ ]:
DEFAULT_SHIP_BINS = (0.1, 0.25, 0.5, 0.75, 1.0)
available_ships = 100
for i, frac in enumerate(DEFAULT_SHIP_BINS):
    print(f'bin {i}: fraction={frac}, ships={max(1, int(available_ships * frac))}')


## 13. Per-Source Decision Loop

Rollout içinde policy tek seferde tek global action seçmiyor. Her valid source planet için ayrı karar alınıyor.

Akış:

```text
source_slots = valid source_mask slotları
for src_slot in source_slots:
    observation copy oluştur
    policy forward
    act_source(out, src_slot)
    decode action
    eğer launch olduysa obs_work'te source ships azaltılır
    target incoming self bucket güncellenir
    source_mask[src_slot] = False
```

Bu `obs_work` güncellemesi önemlidir. Aynı turn içinde bir source'tan launch yapıldıktan sonra sonraki source kararları, önceki launch'ın etkisini yaklaşık olarak görür.


## 14. Scripted Expansion Observation ile Nasıl Etkileşir?

Scripted early neutral expansion, policy actionını ilk `N` turn için override eder ama env action pathini değiştirmez. Yani selected scripted action yine:

```text
obs_decision + src_slot + tgt_slot + ship_bin
  -> decode_action_with_launch
  -> env.step(env_actions)
  -> gerçek fleet launch
  -> reward tracker attribution
```

Scripted action'ın log probability'si `policy.evaluate_source_actions` ile current policy altında hesaplanır. Bu sayede PPO, scripted action'ın gerçek outcome'unu öğrenebilir.


## 15. Observation Değiştirirken Dikkat Edilecekler

Observation feature seti değiştirmek model/checkpoint uyumluluğunu etkiler.

Kurallar:

- `PLANET_FEATURE_DIM`, `FLEET_FEATURE_DIM`, `GLOBAL_FEATURE_DIM`, `EDGE_FEATURE_DIM` değişirse model input layer shape'i değişir. Eski checkpoint yüklenmeyebilir.
- Feature sırası değişirse checkpoint yüklenebilir ama anlam bozulur. Bu çok tehlikelidir.
- Yeni feature eklenecekse en sona eklemek daha güvenlidir.
- `PF_IDX_*` constantları mutlaka güncellenmeli.
- Testlerde shape ve action decode kontrolleri çalıştırılmalı.
- Reward-aware featurelar (`capture_value`, `defense_deficit`) reward değişikliklerinden koparsa policy yanlış öncelik öğrenebilir.

Özellikle production reward exponent gibi reward değişiklikleri observation tarafındaki `capture_value` featureıyla tutarlı olmayabilir. Gerekirse `PF_IDX_CAPTURE_VALUE` de aynı effective production mantığıyla yeniden tasarlanabilir; ama bu checkpoint compatibility kırabilir.
